# STEP 2 — Colab 임베딩 (BAAI/bge-m3 Dense + Sparse)

**순서**
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. 셀 순서대로 실행 (chunks.json은 미리 업로드)
3. `vectors.npz` 다운로드 → 로컬 `data/export/` 에 넣기
4. 로컬에서 `jaehyun_law_upsert.py` 실행

In [ ]:
# 셀 1: GPU 확인
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 셀 2: 패키지 설치
!pip install -q FlagEmbedding

In [ ]:
# 셀 3: chunks.json 업로드 (이미 업로드했으면 스킵)
import os
if os.path.exists('chunks.json'):
    print('chunks.json 이미 존재 — 스킵')
else:
    from google.colab import files
    uploaded = files.upload()

In [ ]:
# 셀 4: 청크 로드 + 중복 제거
import json

with open('chunks.json', encoding='utf-8') as f:
    chunks = json.load(f)

# chunk_id 기준 중복 제거 (마지막 항목 우선)
seen = {}
for c in chunks:
    seen[c['chunk_id']] = c
chunks = list(seen.values())

texts     = [c['text'] for c in chunks]
chunk_ids = [c['chunk_id'] for c in chunks]

print(f'청크 수 (중복 제거 후): {len(chunks)}')
print(f'예시 chunk_id: {chunk_ids[:5]}')

In [ ]:
# 셀 5: BGE-M3 모델 로드
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print('모델 로드 완료')

In [ ]:
# 셀 6: Dense + Sparse 임베딩
import numpy as np

print('임베딩 시작 (dense + sparse)...')
output = model.encode(
    texts,
    batch_size=8,
    max_length=512,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
)

dense_vectors  = output['dense_vecs']        # (N, 1024) float32
sparse_vectors = output['lexical_weights']   # list of dict {token_id: weight}

print(f'dense shape: {dense_vectors.shape}')
print(f'sparse 샘플 키 수: {len(sparse_vectors[0])}개 토큰')

In [ ]:
# 셀 7: vectors.npz 저장
# sparse는 ragged array이므로 object dtype으로 저장
sparse_indices = np.array([list(sv.keys())   for sv in sparse_vectors], dtype=object)
sparse_values  = np.array([list(sv.values()) for sv in sparse_vectors], dtype=object)

np.savez(
    'vectors.npz',
    vectors        = dense_vectors,
    chunk_ids      = np.array(chunk_ids),
    sparse_indices = sparse_indices,
    sparse_values  = sparse_values,
)

print(f'저장 완료!')
print(f'  dense  : {dense_vectors.shape}')
print(f'  sparse : {len(sparse_indices)}개')

In [ ]:
# 셀 8: 다운로드
from google.colab import files
files.download('vectors.npz')